# 04. Graffiti: YOLOv8 detection on STORM

STORM (University of West Attica, CC BY 4.0) gives bounding boxes in CSV files with pixel corners. This notebook converts them to YOLO format and trains `graffiti_model.pt`. The dataset's own split is kept: 813 training and 209 test images, the test split serving as validation.

Our run: 50 epochs, about 29 minutes. Validation box mAP50 0.729, precision 0.836, recall 0.604. Run `00_download_datasets.ipynb` first.

In [ ]:
from pathlib import Path

# Works whether the notebook is opened from ai_model/ or ai_model/notebooks/.
ROOT = Path.cwd().resolve()
while ROOT.name != "ai_model" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert ROOT.name == "ai_model", "Open this notebook from inside the ai_model folder"

DATA = ROOT / "data"          # downloaded and converted datasets (git-ignored)
RAW = DATA / "raw"            # original downloads
RUNS = ROOT / "runs"          # Ultralytics training runs (git-ignored)
WEIGHTS = ROOT / "weights"    # final model files used by the AI service
RESULTS = ROOT / "results"    # metrics used in the project report
for p in (DATA, RAW, RUNS, WEIGHTS, RESULTS):
    p.mkdir(parents=True, exist_ok=True)
print("ai_model root:", ROOT)

In [ ]:
import torch

# Apple Silicon uses the MPS backend; an NVIDIA GPU uses CUDA; otherwise CPU.
DEVICE = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("training device:", DEVICE)

In [ ]:
import csv, collections, shutil

NAME = "graffiti"
storm = RAW / "storm"
out = DATA / NAME
data_yaml = out / "data.yaml"

if not (out / "images" / "train").exists():
    for split, csv_name, img_dir in (("train", "train_labels.csv", "train"), ("val", "test_labels.csv", "test")):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)
        rows = collections.defaultdict(list)
        for r in csv.DictReader(open(storm / "Bounding_boxes" / csv_name)):
            rows[r["filename"]].append(r)
        for fn, boxes in rows.items():
            img = storm / "images" / img_dir / fn
            if not img.exists():
                continue
            shutil.copy(img, out / "images" / split / fn)
            lines = []
            for r in boxes:
                W, H = float(r["width"]), float(r["height"])
                x0, y0, x1, y1 = map(float, (r["xmin"], r["ymin"], r["xmax"], r["ymax"]))
                lines.append(f"0 {(x0 + x1) / 2 / W:.6f} {(y0 + y1) / 2 / H:.6f} {(x1 - x0) / W:.6f} {(y1 - y0) / H:.6f}")
            (out / "labels" / split / f"{Path(fn).stem}.txt").write_text("\n".join(lines) + "\n")
        print(split, len(list((out / "images" / split).glob("*"))), "images")

data_yaml.write_text(f"path: {out}\ntrain: images/train\nval: images/val\nnames: ['Graffiti']\n")

In [ ]:
import shutil
from ultralytics import YOLO


def train(name, data_yaml, task, epochs):
    """Fine-tune a COCO-pretrained YOLOv8 nano model with the settings used for every UrbanFix category."""
    base = "yolov8n-seg.pt" if task == "segment" else "yolov8n.pt"
    model = YOLO(base)
    model.train(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=640,
        batch=16,
        patience=15,          # stop if validation fitness does not improve for 15 epochs
        device=DEVICE,
        seed=0,
        deterministic=False,
        workers=4,
        plots=True,
        project=str(RUNS),
        name=name,
        exist_ok=True,
        verbose=False,
    )
    best = RUNS / name / "weights" / "best.pt"
    target = WEIGHTS / f"{name}_model.pt"
    shutil.copy(best, target)
    print("saved", target)
    return target

In [ ]:
train(NAME, data_yaml, task="detect", epochs=50)

In [ ]:
def quick_val(name, data_yaml):
    """Validate the saved model and print the headline numbers."""
    m = YOLO(str(WEIGHTS / f"{name}_model.pt"))
    r = m.val(data=str(data_yaml), imgsz=640, batch=16, device=DEVICE, plots=False, verbose=False,
              project=str(RUNS / "val"), name=name, exist_ok=True)
    print(f"box  P {r.box.mp:.3f}  R {r.box.mr:.3f}  mAP50 {r.box.map50:.3f}  mAP50-95 {r.box.map:.3f}")
    if hasattr(r, "seg") and r.seg is not None and m.task == "segment":
        print(f"mask P {r.seg.mp:.3f}  R {r.seg.mr:.3f}  mAP50 {r.seg.map50:.3f}  mAP50-95 {r.seg.map:.3f}")
    return r

In [ ]:
quick_val(NAME, data_yaml);